# Демо: `@classmethod`, `@staticmethod`, `@property` и SOLID

Прокликай Shift+Enter каждую ячейку и посмотри, как три декоратора-метода работают по-разному, когда какой выбирать, и как пять SOLID-принципов выглядят в коде на коротких примерах. В конце — три мини-задания.

## Часть 1. Три вида методов

До сих пор мы писали instance-методы — с `self`. Python даёт ещё два декоратора, которые меняют, что именно метод получает первым аргументом.

In [1]:
class Demo:
    def instance_method(self):
        return f"instance — first arg: {type(self).__name__}"

    @classmethod
    def class_method(cls):
        return f"class — first arg: {cls.__name__}"

    @staticmethod
    def static_method():
        return "static — no first arg"

d = Demo()
print(d.instance_method())   # instance — first arg: Demo
print(d.class_method())      # class — first arg: Demo
print(d.static_method())     # static — no first arg

# classmethod и staticmethod можно вызывать прямо через класс — экземпляр не нужен
print(Demo.class_method())   # class — first arg: Demo
print(Demo.static_method())  # static — no first arg

instance — first arg: Demo
class — first arg: Demo
static — no first arg
class — first arg: Demo
static — no first arg


## Часть 2. `@classmethod` — альтернативный конструктор

Самое частое применение — фабрика, которая создаёт объект из нестандартного формата. Внутри пишем `cls(...)` вместо имени класса:

In [2]:
from dataclasses import dataclass

@dataclass
class User:
    name: str
    email: str

    @classmethod
    def from_string(cls, raw):
        # "Аня, anya@x.com" → User(name='Аня', email='anya@x.com')
        name, email = raw.split(",")
        return cls(name=name.strip(), email=email.strip())

    @classmethod
    def from_dict(cls, data):
        return cls(name=data["name"], email=data["email"])

u1 = User.from_string("Аня, anya@x.com")
u2 = User.from_dict({"name": "Боря", "email": "borya@x.com"})
print(u1)              # User(name='Аня', email='anya@x.com')
print(u2)              # User(name='Боря', email='borya@x.com')

User(name='Аня', email='anya@x.com')
User(name='Боря', email='borya@x.com')


Главное преимущество `cls` над прямым вызовом `User(...)` — **корректная работа при наследовании**. Создадим наследника и позовём ту же фабрику через него:

In [3]:
@dataclass
class Admin(User):
    pass     # наследник без своих полей, ради демонстрации

# Тот же метод, разные классы — разные результаты
u = User.from_string("Аня, a@x.com")
a = Admin.from_string("Боря, b@x.com")

print(type(u).__name__)   # User
print(type(a).__name__)   # Admin — cls подставился правильно

# Если бы внутри from_string было жёстко User(...), то Admin.from_string
# тоже вернул бы User. Это потеря смысла фабрики у наследника.

User
Admin


Это та же причина, по которой в HuggingFace `BertModel.from_pretrained(...)` возвращает `BertModel`, а `GPT2Model.from_pretrained(...)` возвращает `GPT2Model` — метод объявлен в базовом классе один раз, но через `cls` правильно работает на любом наследнике.

## Часть 3. `@staticmethod` — утилита внутри класса

Если функция логически связана с классом, но не использует ни состояние объекта, ни сам класс — её можно сделать `@staticmethod`. Это просто обычная функция, которая лежит в namespace класса для организации кода. На практике нужен редко.

In [4]:
class TemperatureConverter:
    @staticmethod
    def celsius_to_fahrenheit(c):
        return c * 9 / 5 + 32

    @staticmethod
    def fahrenheit_to_celsius(f):
        return (f - 32) * 5 / 9

print(TemperatureConverter.celsius_to_fahrenheit(100))   # 212.0
print(TemperatureConverter.fahrenheit_to_celsius(32))    # 0.0

212.0
0.0


Альтернатива — функции на уровне модуля рядом с классом. Часто именно так и делают. `@staticmethod` берут только когда вызовы через `ClassName.func(...)` улучшают читаемость.

## Часть 4. `@property` — атрибут с логикой за фасадом

Иногда хочется, чтобы при чтении атрибута срабатывал какой-то код (например, вычисление производного значения), а синтаксис выглядел как обычный атрибут — без скобок. Для этого есть `@property`.

In [5]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):
        return 3.14 * self.radius ** 2

c = Circle(5)
print(c.area)         # 78.5 — вызвался метод area, но без скобок
c.radius = 10
print(c.area)         # 314.0 — пересчиталось при следующем доступе

78.5
314.0


Можно добавить и сеттер — тогда при `obj.attr = value` сработает ваш код. Типичное применение — валидация:

In [6]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius        # сразу через сеттер — валидация сработает

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError(f"{value} ниже абсолютного нуля")
        self._celsius = value

t = Temperature(20)
print(t.celsius)     # 20
t.celsius = 100
print(t.celsius)     # 100

try:
    t.celsius = -500    # сработает валидация в сеттере
except ValueError as e:
    print(f"ValueError: {e}")

20
100
ValueError: -500 ниже абсолютного нуля


## Часть 5. SOLID — S: Single Responsibility (одна ответственность на класс)

Класс должен решать ровно одну задачу. Если класс одновременно загружает данные, чистит их, обучает модель и пишет логи — это четыре ответственности. Любое изменение в одной из них может сломать остальные. Пример:

In [7]:
# Плохо — один класс отвечает за всё
class Pipeline:
    def load(self, path):
        return f"loaded {path}"
    def clean(self, data):
        return f"cleaned {data}"
    def train(self, data):
        return f"trained on {data}"
    def save_log(self, message):
        print(f"LOG: {message}")

# Лучше — четыре класса, каждый со своей зоной ответственности
class DataLoader:
    def load(self, path):
        return f"loaded {path}"

class DataCleaner:
    def clean(self, data):
        return f"cleaned {data}"

class Trainer:
    def train(self, data):
        return f"trained on {data}"

class Logger:
    def save_log(self, message):
        print(f"LOG: {message}")

loader = DataLoader()
cleaner = DataCleaner()
trainer = Trainer()
logger = Logger()

raw = loader.load("data.csv")
clean = cleaner.clean(raw)
model = trainer.train(clean)
logger.save_log(model)

LOG: trained on cleaned loaded data.csv


## Часть 6. SOLID — O: Open/Closed (открыт к расширению, закрыт к модификации)

Новое поведение должно добавляться **расширением** существующего кода, а не его изменением. Классический контр-пример — функция с цепочкой `if shape.type == ...`:

In [8]:
# Плохо — каждая новая фигура требует менять функцию area
def area_bad(shape):
    if shape["type"] == "circle":
        return 3.14 * shape["radius"] ** 2
    elif shape["type"] == "square":
        return shape["side"] ** 2
    # завтра добавим треугольник — снова трогать area_bad
    raise ValueError("unknown shape")

shapes = [{"type": "circle", "radius": 5}, {"type": "square", "side": 3}]
for s in shapes:
    print(area_bad(s))

78.5
9


In [9]:
# Лучше — каждая фигура свой класс с методом area; функция-консьюмер
# работает с любым наследником Shape без изменений
class Shape:
    def area(self):
        raise NotImplementedError

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    def area(self):
        return 3.14 * self.radius ** 2

class Square(Shape):
    def __init__(self, side):
        self.side = side
    def area(self):
        return self.side ** 2

# Добавляем третью фигуру — НИЧЕГО НЕ ТРОГАЕМ из предыдущего кода
class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height
    def area(self):
        return self.base * self.height / 2

shapes = [Circle(5), Square(3), Triangle(4, 6)]
for s in shapes:
    print(f"{type(s).__name__}: {s.area()}")

Circle: 78.5
Square: 9
Triangle: 12.0


## Часть 7. SOLID — L, I, D коротко

Оставшиеся три принципа важны для крупных систем; в ML-коде явно обсуждаются реже. Покажем по одному короткому снипету каждый.

In [10]:
# L — Liskov Substitution: дочерний работает там, где ожидается родительский, без сюрпризов
class User:
    def can_read(self, doc):
        return True

class Admin(User):
    def can_read(self, doc):
        return True   # Admin не урезает контракт User'а

def render_doc(user, doc):
    if user.can_read(doc):
        return f"showing {doc}"
    return "forbidden"

# Любая функция, ожидающая User, должна корректно работать с Admin
print(render_doc(User(), "report"))     # showing report
print(render_doc(Admin(), "report"))    # showing report — без сюрпризов

showing report
showing report


In [11]:
# I — Interface Segregation: лучше несколько маленьких интерфейсов, чем один большой
# В Python без формальных интерфейсов это значит: не делай базовый класс
# с 15 абстрактными методами, которые наследники потом игнорируют.

# Хорошо — два маленьких "интерфейса" (классы-протоколы)
class Readable:
    def read(self): raise NotImplementedError

class Writable:
    def write(self, data): raise NotImplementedError

# Класс берёт только нужные интерфейсы
class ReadOnlyFile(Readable):
    def read(self):
        return "file contents"

class FullFile(Readable, Writable):
    def read(self):
        return "contents"
    def write(self, data):
        print(f"writing {data}")

ro = ReadOnlyFile()
print(ro.read())
# ReadOnlyFile.write() вообще нет — это и есть segregation

file contents


In [12]:
# D — Dependency Inversion: зависим от абстракций, не от конкретных классов
# Это позволяет в тестах подменить настоящую базу на заглушку.

class Database:
    def get_user(self, user_id):
        return f"real user #{user_id} from DB"

class FakeDatabase:    # подмена для тестов
    def get_user(self, user_id):
        return f"fake user #{user_id}"

class UserService:
    def __init__(self, db):           # зависит от АБСТРАКЦИИ "что-то с get_user"
        self.db = db
    def show(self, user_id):
        return self.db.get_user(user_id)

# В проде — настоящая база, в тестах — fake
service = UserService(Database())
print(service.show(42))             # real user #42 from DB

test_service = UserService(FakeDatabase())
print(test_service.show(42))        # fake user #42

real user #42 from DB
fake user #42


## Мини-задания

Три коротких упражнения. Подсказок к именам и методам нет — вспомни сам.

**Задание 1.** Напиши `@dataclass class Url` с полями `scheme: str`, `host: str`, `path: str`. Добавь `@classmethod`, который парсит строку вида `"https://example.com/api/v1"` и возвращает объект `Url`.

**Задание 2.** Напиши класс `Rectangle` с полями `width` и `height`. Через `@property` сделай вычисляемый атрибут `area`, который возвращает `width * height`. Доступ к `area` — без скобок, как к обычному атрибуту.

**Задание 3.** Что напечатает код ниже? Сначала угадай, потом запусти.

In [13]:
# Задание 1
# from dataclasses import dataclass
#
# @dataclass
# class Url:
#     ...

# Проверка:
# u = Url.from_string("https://example.com/api/v1")
# print(u)   # Url(scheme='https', host='example.com', path='/api/v1')


In [14]:
# Задание 2
# class Rectangle:
#     ...

# Проверка:
# r = Rectangle(3, 4)
# print(r.area)   # 12 — без скобок
# r.width = 10
# print(r.area)   # 40 — пересчитался


In [15]:
# Задание 3 — твой прогноз для каждой строки впиши в комментарий:
class Animal:
    @classmethod
    def kind(cls):
        return cls.__name__

class Dog(Animal):
    pass

print(Animal.kind())  # ?
print(Dog.kind())     # ?
print(Dog().kind())   # ?


Animal
Dog
Dog
